In [1]:
# System and environment variables
import os
from dotenv import load_dotenv
import zipfile

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization and analysis
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine Learning models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation metrics
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
import os
from dotenv import load_dotenv

# Loading the environment variables containing the Kaggle API token
load_dotenv('Kaggle_token.env')

# Importing the Kaggle API explicitly after loading the environment variables
from kaggle.api.kaggle_api_extended import KaggleApi

# Initializing and authenticating the Kaggle API client
api = KaggleApi()
api.authenticate()

# Downloading and extracting the dataset directly into the current working directory
print("Downloading and extracting dataset...")
api.dataset_download_files('prachi13/customer-analytics', path='.', force=True, unzip=True)

print("Dataset successfully downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/prachi13/customer-analytics
Dataset successfully downloaded and extracted.


In [3]:
# Reading the file
ca = pd.read_csv('train.csv')

In [4]:
# View of the first 5 rows
ca.head()

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [5]:
# Statistical information about the data
ca.describe()

,ID,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
count,10999.00000,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000,10999.000000
mean,5500.00000,4.054459,2.990545,210.196836,3.567597,13.373216,3634.016729,0.596691
std,3175.28214,1.141490,1.413603,48.063272,1.522860,16.205527,1635.377251,0.490584
min,1.00000,2.000000,1.000000,96.000000,2.000000,1.000000,1001.000000,0.000000
25%,2750.50000,3.000000,2.000000,169.000000,3.000000,4.000000,1839.500000,0.000000
50%,5500.00000,4.000000,3.000000,214.000000,3.000000,7.000000,4149.000000,1.000000
75%,8249.50000,5.000000,4.000000,251.000000,4.000000,10.000000,5050.000000,1.000000
max,10999.00000,7.000000,5.000000,310.000000,10.000000,65.000000,7846.000000,1.000000


In [6]:
ca.drop('ID', axis = 1, inplace = True)

In [7]:
# Mode of shipment by reached on Time
ca.groupby('Mode_of_Shipment')['Reached.on.Time_Y.N'].mean()

Mode_of_Shipment
Flight    0.601576
Road      0.588068
Ship      0.597561
Name: Reached.on.Time_Y.N, dtype: float64

In [8]:
# Converting categorical variables into numeric dummy variables
ca = pd.get_dummies(ca, drop_first=True)

In [9]:
# Separating features (X) and target (y)
X = ca.drop('Reached.on.Time_Y.N', axis=1)
y = ca['Reached.on.Time_Y.N']

# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Normalizing the features to avoid scale bias
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
# Initializing, training, and predicting with the model
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

In [11]:
# Initializing, training, and predicting with the model
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

In [12]:
# --- Model Evaluation ---
# Comparing model performances using classification reports to evaluate Precision, Recall, and F1-Score
print("--- K-NN Performance ---")
print(classification_report(y_test, y_pred_knn))

print("\n--- SVM Performance ---")
print(classification_report(y_test, y_pred_svm))

--- K-NN Performance ---
              precision    recall  f1-score   support

           0       0.53      0.60      0.56      1312
           1       0.71      0.65      0.68      1988

    accuracy                           0.63      3300
   macro avg       0.62      0.62      0.62      3300
weighted avg       0.64      0.63      0.63      3300


--- SVM Performance ---
              precision    recall  f1-score   support

           0       0.56      0.74      0.64      1312
           1       0.78      0.61      0.69      1988

    accuracy                           0.67      3300
   macro avg       0.67      0.68      0.66      3300
weighted avg       0.69      0.67      0.67      3300



In [15]:
# Importing the heavy artillery: Ensemble models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# --- 1. Random Forest Model ---
# Training a Random Forest with 100 decision trees to capture non-linear patterns
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predicting the test set results
y_pred_rf = rf.predict(X_test)

# --- 2. Gradient Boosting Model ---
# Training a Gradient Boosting model to sequentially correct previous errors
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

# Predicting the test set results
y_pred_gb = gb.predict(X_test)

# --- 3. XGBoost Model ---
# Training the XGBoost model, the industry standard for complex tabular data
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

# Predicting the test set results
y_pred_xgb = xgb.predict(X_test)

# --- 4. The Ultimate Evaluation ---
print("--- Random Forest Performance ---")
print(classification_report(y_test, y_pred_rf))

print("\n--- Gradient Boosting Performance ---")
print(classification_report(y_test, y_pred_gb))

print("\n--- XGBoost Performance ---")
print(classification_report(y_test, y_pred_xgb))

--- Random Forest Performance ---
              precision    recall  f1-score   support

           0       0.56      0.69      0.62      1312
           1       0.76      0.64      0.69      1988

    accuracy                           0.66      3300
   macro avg       0.66      0.67      0.66      3300
weighted avg       0.68      0.66      0.66      3300


--- Gradient Boosting Performance ---
              precision    recall  f1-score   support

           0       0.56      0.88      0.68      1312
           1       0.87      0.54      0.67      1988

    accuracy                           0.68      3300
   macro avg       0.71      0.71      0.68      3300
weighted avg       0.74      0.68      0.67      3300


--- XGBoost Performance ---
              precision    recall  f1-score   support

           0       0.56      0.77      0.65      1312
           1       0.80      0.60      0.69      1988

    accuracy                           0.67      3300
   macro avg       0.68   

C:\Users\Guilh\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:05:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [16]:
# Exporting the cleaned e-commerce dataset for Power BI visualization
ca.to_csv('ecommerce_cleaned.csv', index=False)